In [2]:
!git clone https://github.com/Sasha15151/MLF-Skin-Cancer-Classification.git

Cloning into 'MLF-Skin-Cancer-Classification'...
remote: Enumerating objects: 89, done.
remote: Counting objects: 100% (89/89), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 89 (delta 21), reused 42 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (89/89), 24.49 KiB | 964.00 KiB/s, done.
Resolving deltas: 100% (21/21), done.


In [3]:
%cd MLF-Skin-Cancer-Classification
!ls

/content/MLF-Skin-Cancer-Classification
notebooks  README.md  requirements.txt


In [4]:
!pip install -q -r requirements.txt

# Data Loading

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
%cd /content/drive/MyDrive/Skin Cancer Data

/content/drive/MyDrive/Skin Cancer Data


# Data Normalization

In [7]:
import pandas as pd

df = pd.read_csv("HAM10000_metadata.tab", sep="\t")

label_dict = {label: idx for idx, label in enumerate(df["dx"].unique())}
df["label"] = df["dx"].map(label_dict)

print(label_dict)

{'bkl': 0, 'nv': 1, 'df': 2, 'mel': 3, 'vasc': 4, 'bcc': 5, 'akiec': 6}


In [8]:
import os
from PIL import Image
from torch.utils.data import Dataset

class SkinCancerDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df = dataframe
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]["image_id"] + ".jpg"
        label = self.df.iloc[idx]["label"]

        img_path = os.path.join(self.image_dir, img_name)
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [9]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

In [10]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [11]:
from torch.utils.data import DataLoader

image_dir = "HAM10000_images"

train_dataset = SkinCancerDataset(train_df, image_dir, transform)
val_dataset   = SkinCancerDataset(val_df, image_dir, transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [12]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([32, 3, 224, 224])
torch.Size([32])


# Split Dataset

In [15]:
# Split the test set again in a validation set and a true test set:
val_df, test_df = train_test_split(val_df, test_size=0.5)
train_df = train_df.reset_index()
val_df = val_df.reset_index()
test_df = test_df.reset_index()

In [16]:
print(len(train_df))
print(len(val_df))
print(len(test_df))

8012
1001
1002


In [21]:
train_df["label"].value_counts()

,count
label,
1,5364
3,890
0,879
5,411
6,262
4,114
2,92
